# MedXIR corpus: chunk every unique text

Input: the `medxir-corpus` Kaggle dataset (no internet needed). Output in `/kaggle/working`:
`chunks/<lang>/part-NNN.parquet`, `hash_groups.parquet` and `chunk_manifest.json`.

The cells that define `chunk_document`, `clean_text` and `hash_groups` are copies of
`scripts/chunking.py`, `scripts/text_clean.py` and `scripts/dedup_groups.py`.

In [ ]:
# Copied from chunking.py by scripts/build_chunk_notebook.py. Edit the script and rebuild.
"""Paragraph-first chunking with exact source offsets.

Every chunk is text[start:end] of the exported document, so chunk_text is always
a verbatim extract (SUBMISSION.md). Size is counted in words, or in characters/2
for Chinese. The scorer divides overlap by the *reference* chunk length, so a
chunk shorter than 40% of the reference can never match; err on the large side.

    uv run python scripts/chunking.py stats artifacts/corpus_v1/en.parquet --limits 120 250 400
"""
import argparse
import re
from dataclasses import dataclass

PARAGRAPH = re.compile(r'\n\s*\n')
SENTENCE_END = re.compile(r'(?<=[.!?。！？；;])\s+|(?<=[。！？；])')


@dataclass
class Chunk:
    order: int
    start: int
    end: int
    text: str


def size(text, lang):
    return len(text) // 2 if lang == 'zh' else len(text.split())


def spans(text, pattern, start=0, end=None):
    """Non-empty (start, end) spans of text[start:end] split by pattern, whitespace-trimmed."""
    end = len(text) if end is None else end
    cursor = start
    for match in list(pattern.finditer(text, start, end)) + [None]:
        stop = match.start() if match else end
        piece = text[cursor:stop]
        if piece.strip():
            left = cursor + len(piece) - len(piece.lstrip())
            right = cursor + len(piece.rstrip())
            yield left, right
        if match:
            cursor = match.end()


def sentences(text, start, end, lang, limit):
    """Sentence spans; a sentence above the limit (tables, lists without punctuation)
    is cut at line breaks, then hard-cut at whitespace (or every 2*limit chars for zh)."""
    for s_start, s_end in spans(text, SENTENCE_END, start, end):
        if size(text[s_start:s_end], lang) <= limit:
            yield s_start, s_end
            continue
        for l_start, l_end in spans(text, re.compile(r'\n'), s_start, s_end):
            if size(text[l_start:l_end], lang) <= limit:
                yield l_start, l_end
            elif lang == 'zh':
                for cut in range(l_start, l_end, 2 * limit):
                    yield cut, min(cut + 2 * limit, l_end)
            else:
                words = [m.span() for m in re.finditer(r'\S+', text[l_start:l_end])]
                for i in range(0, len(words), limit):
                    piece = words[i:i + limit]
                    yield l_start + piece[0][0], l_start + piece[-1][1]


def units(text, lang, limit):
    """Paragraphs, with paragraphs above the limit split into sentence groups."""
    for start, end in spans(text, PARAGRAPH):
        if size(text[start:end], lang) <= limit:
            yield start, end
            continue
        group = None
        for s_start, s_end in sentences(text, start, end, lang, limit):
            if group and size(text[group[0]:s_end], lang) > limit:
                yield group
                group = None
            group = (group[0], s_end) if group else (s_start, s_end)
        if group:
            yield group


def is_heading(text, lang):
    """Short line without closing punctuation: a section title or list label."""
    text = text.strip()
    short = len(text) <= 30 if lang == 'zh' else len(text.split()) <= 12
    return short and not text.endswith(('.', '!', '?', ':', ';', '。', '！', '？', '：', '；', '"', '”', ')'))


def chunk_document(text, lang, limit=350, minimum=40, overlap=50):
    """Greedily merge adjacent units up to `limit`; fold a tiny tail into the previous chunk.

    Headings never end a chunk: trailing headings move to the next chunk, so a title
    always stays with the text it introduces. A chunk that continues the same section
    also repeats the previous chunk's last whole sentences (up to `overlap` units);
    a chunk opening on a heading starts a new section and gets no overlap.
    """
    groups = []
    for unit in units(text, lang, limit):
        if groups and size(text[groups[-1][0][0]:unit[1]], lang) <= limit:
            groups[-1].append(unit)
            continue
        carried = []
        while groups and len(groups[-1]) > 1 and is_heading(text[groups[-1][-1][0]:groups[-1][-1][1]], lang):
            carried.insert(0, groups[-1].pop())
        groups.append(carried + [unit])
    merged = [(group[0][0], group[-1][1]) for group in groups]
    if len(merged) > 1 and size(text[merged[-1][0]:merged[-1][1]], lang) < minimum:
        tail = merged.pop()
        merged[-1] = (merged[-1][0], tail[1])
    chunks = []
    for order, (start, end) in enumerate(merged):
        if order and overlap and not is_heading(text[start:end].split('\n', 1)[0], lang):
            previous_start, previous_end = merged[order - 1]
            for s_start, _ in reversed(list(spans(text, SENTENCE_END, previous_start, previous_end))):
                if size(text[s_start:previous_end], lang) > overlap:
                    break
                start = s_start
        chunks.append(Chunk(order, start, end, text[start:end]))
    return chunks


def sample_rows(path, sample):
    """(lang, text) of a random sample, read one row group at a time: take() on the whole
    text column overflows the 2 GB string offsets of the 1 GB vi corpus."""
    import numpy as np
    import pyarrow.parquet as pq

    file = pq.ParquetFile(path)
    total = file.metadata.num_rows
    wanted = None
    if sample and total > sample:
        wanted = set(np.random.default_rng(0).choice(total, sample, replace=False).tolist())
    rows, offset = [], 0
    for group in range(file.metadata.num_row_groups):
        count = file.metadata.row_group(group).num_rows
        columns = file.read_row_group(group, columns=['lang', 'text']).to_pydict()
        rows += [(columns['lang'][i], columns['text'][i]) for i in range(count)
                 if wanted is None or offset + i in wanted]
        offset += count
    return rows


def stats(path, limits, sample):
    import numpy as np

    rows = sample_rows(path, sample)
    langs, texts = [lang for lang, _ in rows], [text for _, text in rows]
    lengths = np.array([size(t, l) for t, l in zip(texts, langs)])
    unit = 'chars/2' if langs and langs[0] == 'zh' else 'words'
    print(f'{path}: {len(texts):,} documents sampled, length in {unit}')
    print('  document length p10/p50/p90/p99:', *np.percentile(lengths, [10, 50, 90, 99]).astype(int))
    for limit in limits:
        counts, chunk_sizes = [], []
        for text, lang in zip(texts, langs):
            chunks = chunk_document(text, lang, limit)
            counts.append(len(chunks))
            chunk_sizes.extend(size(c.text, lang) for c in chunks)
            assert all(text[c.start:c.end] == c.text for c in chunks)
        scale = len(texts) if not sample else None
        print(f'  limit {limit:>4}: chunks/doc mean {np.mean(counts):.2f}, '
              f'chunk size p50/p90 {int(np.percentile(chunk_sizes, 50))}/{int(np.percentile(chunk_sizes, 90))}, '
              f'single-chunk docs {np.mean(np.array(counts) == 1):.0%}'
              + (f', total {sum(counts):,}' if scale else ''))


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('action', choices=['stats'])
    parser.add_argument('parquet')
    parser.add_argument('--limits', type=int, nargs='+', default=[120, 250, 400])
    parser.add_argument('--sample', type=int, default=20000, help='0 = all documents')
    args = parser.parse_args()
    stats(args.parquet, args.limits, args.sample)

In [ ]:
# Copied from text_clean.py by scripts/build_chunk_notebook.py. Edit the script and rebuild.
"""Source-text cleaning, applied before chunking. Every chunk row carries CLEAN_VERSION.

1. Line-break tags become line breaks: <br>, <br/>, <br />, &lt;br&gt;, and the fullwidth
   ＜br＞ / ＜br ／＞ found in about 1,077 zh documents. A run of tags (with the whitespace
   around it) becomes as many newlines as it had tags or existing newlines, capped at one blank
   line, so a break next to a paragraph break does not add a second blank line.
2. zh lead-in: a first paragraph starting with 核心提示 is dropped when the rest of the document
   repeats its text, because the site shows a summary before the article. A lead-in with no
   text is dropped too. A standalone summary that does not reappear is kept.

    核心提示：婴儿泪囊炎 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病……\n\n婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……
    -> 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……

Nothing else changes (no Unicode or whitespace normalisation).

    from text_clean import clean_text
"""
import re

CLEAN_VERSION = 'clean-v1'

BR_TAG = r'(?:<br\s*/?>|＜br\s*[/／]?\s*＞|&lt;br\s*/?&gt;)'
BR_RUN = re.compile(rf'[ \t　\n]*(?:{BR_TAG}[ \t　\n]*)+', re.IGNORECASE)
BR_ONE = re.compile(BR_TAG, re.IGNORECASE)
LEAD_IN = re.compile(r'^(?:核心提示[ \t　]*[:：][ \t　]*)+')
PARAGRAPH_BREAK = re.compile(r'\n\s*\n')


def line_break(match):
    run = match.group(0)
    return '\n' * min(max(len(BR_ONE.findall(run)), run.count('\n')), 2)


def drop_lead_in(text):
    """Drop a top 核心提示 paragraph whose first 40 characters (whitespace removed, trailing
    ellipsis removed) share a 10-character run with the rest of the document."""
    if not LEAD_IN.match(text):
        return text
    first, *rest = PARAGRAPH_BREAK.split(text, maxsplit=1)
    rest_text = rest[0] if rest else ''
    body = re.sub(r'\s+', '', LEAD_IN.sub('', first)).rstrip('….')
    if not body:
        return rest_text
    head, rest_plain = body[:40], re.sub(r'\s+', '', rest_text)
    if any(head[i:i + 10] in rest_plain for i in range(max(1, len(head) - 9))):
        return rest_text
    return text


def clean_text(text, lang):
    text = BR_RUN.sub(line_break, text)
    if lang == 'zh':
        text = drop_lead_in(text)
    return text

In [ ]:
# Copied from dedup_groups.py by scripts/build_chunk_notebook.py. Edit the script and rebuild.
"""Exact-duplicate groups of the Kaggle corpus, without the Pharmacity bad redirects.

pharmacity_exclusions(): IDs whose fetched page is another article (the redirect ended on a page
without the original slug), so their text is wrong for their doc_id.
hash_groups(): one row per text_sha256 among in_corpus documents that are not excluded.

group_type: single; url_variant (same host and path, ignoring scheme, www, trailing slash and
fragment); cross_host (more than one host); same_host_diff_path (redirects to a merged article,
republished abstracts, placeholder pages). Only single and url_variant are fan-out safe: the
chunks of the representative can be reported for every doc_id of the group.

    uv run python scripts/dedup_groups.py --registry artifacts/kaggle/medxir-corpus/registry.parquet
"""
import argparse
import gzip
import json
from pathlib import Path
from urllib.parse import unquote, urlsplit

import pandas as pd

REASON = 'redirected_to_other_article'


def pharmacity_exclusions(jsonl_gz_path):
    """doc_id and reason of each record whose final_url path lacks the original slug."""
    rows = []
    with gzip.open(jsonl_gz_path, 'rt', encoding='utf-8') as file:
        for line in file:
            record = json.loads(line)
            slug = unquote(urlsplit(record['url']).path.strip('/'))
            landed = unquote(urlsplit(record['final_url']).path)
            if slug not in landed:
                rows.append((record['doc_id'], REASON))
    return pd.DataFrame(rows, columns=['doc_id', 'reason']).sort_values('doc_id').reset_index(drop=True)


def url_key(url):
    parts = urlsplit(url)
    return parts.netloc.lower().removeprefix('www.') + parts.path.rstrip('/').lower()


def hash_groups(registry, excluded_ids=()):
    """One row per text_sha256. rep_doc_id is the smallest doc_id: duplicate texts are identical,
    and no crawl-vs-archive reason favours one source, so the choice only sets url/title metadata."""
    rows = registry[registry.state.eq('in_corpus') & ~registry.doc_id.isin(list(excluded_ids))]
    rows = rows[['doc_id', 'url', 'host', 'text_sha256']].sort_values('doc_id')
    rows = rows.assign(key=rows.url.map(url_key))
    grouped = rows.groupby('text_sha256', sort=True)
    summary = pd.DataFrame({
        'rep_doc_id': grouped.doc_id.min(),
        'doc_ids': grouped.doc_id.agg(lambda ids: [int(i) for i in ids]),
        'n_ids': grouped.size(),
        'n_keys': grouped.key.nunique(),
        'n_hosts': grouped.host.nunique(),
    }).reset_index()
    group_type = pd.Series('same_host_diff_path', index=summary.index)
    group_type = group_type.mask(summary.n_hosts > 1, 'cross_host')
    group_type = group_type.mask(summary.n_keys == 1, 'url_variant')
    group_type = group_type.mask(summary.n_ids == 1, 'single')
    summary['group_type'] = group_type
    summary['fanout_safe'] = summary.group_type.isin(['single', 'url_variant'])
    return summary[['text_sha256', 'rep_doc_id', 'doc_ids', 'n_ids', 'group_type', 'fanout_safe']]


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--registry', type=Path, default=Path('artifacts/kaggle/medxir-corpus/registry.parquet'))
    parser.add_argument('--pharmacity', type=Path, default=Path('artifacts/recrawl_20261015/pharmacity_v1/documents.jsonl.gz'))
    parser.add_argument('--output', type=Path, default=Path('artifacts/aux_v1'))
    args = parser.parse_args()

    args.output.mkdir(parents=True, exist_ok=True)
    exclusions = pharmacity_exclusions(args.pharmacity)
    exclusions.to_csv(args.output / 'exclusions.csv', index=False)
    registry = pd.read_parquet(args.registry)
    groups = hash_groups(registry, exclusions.doc_id)
    groups.to_parquet(args.output / 'hash_groups.parquet', index=False)

    lang_of = registry.set_index('doc_id').lang
    lang = groups.rep_doc_id.map(lang_of)
    print(f'excluded ids: {len(exclusions):,} {exclusions.reason.value_counts().to_dict()}')
    print(f'hash groups: {len(groups):,}; documents covered: {int(groups.n_ids.sum()):,}')
    print('\ngroups by type and lang of the representative:')
    print(pd.crosstab(groups.group_type, lang, margins=True))
    rows = groups.assign(lang=lang).groupby(['group_type', 'lang']).n_ids.sum().unstack(fill_value=0)
    print('\ndocument ids by type and lang (duplicates only, singles included):')
    print(rows.assign(total=rows.sum(axis=1)))

In [ ]:
EXCLUDED_DOC_IDS = [
    317708, 317789, 317794, 317800, 317804, 317826, 317828, 317865, 318425, 318545,
    318547, 318573, 318615, 318618, 318636, 318637, 318638, 318641, 318649, 318656,
    318672, 318698, 318709, 318934, 318950, 318967, 319016, 319052, 319083, 319149,
    319164, 319178, 319281, 319310, 319316, 319338, 319374, 319440, 319441, 319467,
    319582, 319602, 320233, 320286, 320431, 320465, 320525, 320526, 320527, 320529,
    320532, 320536, 320557, 320558, 320598, 320723, 320748, 320751, 320854, 320855,
    320856, 320857, 320858, 320859, 320951, 320958, 321019, 321092, 321093, 321212,
    321291, 321292, 321293, 321294, 321295, 321299, 321300, 321301, 321382, 321383,
    321389, 321391, 321392, 321398, 321405, 321409, 321473, 321636, 321720, 321722,
    321723, 321724, 321758, 321816, 321850, 321860, 321886, 321920, 322027, 322091,
    322094, 322096, 322121, 322159, 322184, 322268, 322278, 322301, 322302, 322314,
    322360, 322757, 322758, 322960, 323019, 323020, 323028, 323029,
]  # scripts/dedup_groups.py pharmacity_exclusions()

In [ ]:
import csv
import glob
import inspect
import json
import os
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Local test runs set DATA_ROOT, SAMPLE_DOCS and OUT_DIR. On Kaggle they stay unset.
DATA_ROOT = os.environ.get('DATA_ROOT', '')
SAMPLE_DOCS = int(os.environ.get('SAMPLE_DOCS', '0'))  # documents per part; 0 = all
OUT = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
WORKERS = 4
PARAMS = inspect.signature(chunk_document).parameters
CHUNK_PARAMS = '/'.join(str(PARAMS[name].default) for name in ('limit', 'overlap', 'minimum'))
COLUMNS = ['chunk_id', 'text_sha256', 'rep_doc_id', 'lang', 'order', 'start', 'end',
           'chunk_text', 'n_units', 'title', 'clean_version', 'chunk_params']
SCHEMA = pa.schema([
    ('chunk_id', pa.string()), ('text_sha256', pa.string()), ('rep_doc_id', pa.int64()),
    ('lang', pa.string()), ('order', pa.int32()), ('start', pa.int64()), ('end', pa.int64()),
    ('chunk_text', pa.string()), ('n_units', pa.int32()), ('title', pa.string()),
    ('clean_version', pa.string()), ('chunk_params', pa.string()),
])
print('chunk_params', CHUNK_PARAMS, '| clean_version', CLEAN_VERSION, '| workers', WORKERS)

## 1. Dataset root and manifest

In [ ]:
def find_root():
    if DATA_ROOT:
        return Path(DATA_ROOT)
    hits = sorted(glob.glob('/kaggle/input/**/registry.parquet', recursive=True))
    assert len(hits) == 1, hits
    return Path(hits[0]).parent


T0 = time.time()
ROOT = find_root()
MANIFEST = json.loads((ROOT / 'manifest.json').read_text())
OUT.mkdir(parents=True, exist_ok=True)
print('root', ROOT, '| version', MANIFEST['version'], '| built', MANIFEST['built_at'],
      '| documents', f"{MANIFEST['documents']:,}")

## 2. Hash groups (exclusions applied)

In [ ]:
registry = pd.read_parquet(ROOT / 'registry.parquet')
groups = hash_groups(registry, EXCLUDED_DOC_IDS)
groups.to_parquet(OUT / 'hash_groups.parquet', index=False)
del registry
print(f'excluded {len(EXCLUDED_DOC_IDS)}; hash groups {len(groups):,}; '
      f'documents covered {int(groups.n_ids.sum()):,}')
print(groups.group_type.value_counts().to_string())

## 3. Clean and chunk, part by part

Each worker cleans one unique text, chunks it with `chunk_document`, and asserts that every chunk
is `cleaned[start:end]`. `not_in_raw` counts chunks that are not a substring of the exported text,
which happens only where cleaning changed the text (line-break tags, lead-in).

In [ ]:
def chunk_one(job):
    text_sha256, rep_doc_id, lang, title, raw = job
    cleaned = clean_text(raw, lang)
    chunks = chunk_document(cleaned, lang)
    for chunk in chunks:
        assert cleaned[chunk.start:chunk.end] == chunk.text, (rep_doc_id, chunk.order)
    return dict(
        text_sha256=text_sha256, rep_doc_id=rep_doc_id, title=title,
        changed=cleaned != raw, not_in_raw=sum(chunk.text not in raw for chunk in chunks),
        chunks=[(chunk.order, chunk.start, chunk.end, chunk.text, size(chunk.text, lang))
                for chunk in chunks])


REP_SHA = dict(zip(groups.rep_doc_id, groups.text_sha256))
IDS_OF_SHA = dict(zip(groups.text_sha256, groups.n_ids))
REP_IDS = groups.rep_doc_id.values
PER_LANG = {}

In [ ]:
def clean_title(value):
    return None if pd.isna(value) else str(value)


def write_part(lang, number, rows):
    columns = dict(zip(COLUMNS, map(list, zip(*rows)))) if rows else {name: [] for name in COLUMNS}
    table = pa.table(columns, schema=SCHEMA)
    assert len(set(columns['chunk_id'])) == table.num_rows, 'chunk_id is not unique'
    folder = OUT / 'chunks' / lang
    folder.mkdir(parents=True, exist_ok=True)
    pq.write_table(table, folder / f'part-{number:03d}.parquet', compression='zstd')
    return table.num_rows


with Pool(WORKERS) as pool:
    for lang in ('en', 'vi', 'zh'):
        stats = dict(texts=0, doc_ids=0, changed=0, changed_ids=0, not_in_raw=0,
                     chunk_counts=[], units=[], chunks=0, read_seconds=0.0, chunk_seconds=0.0,
                     write_seconds=0.0)
        PER_LANG[lang] = stats
        for number, path in enumerate(sorted((ROOT / 'docs' / lang).glob('*.parquet'))):
            started = time.time()
            docs = pd.read_parquet(path, columns=['doc_id', 'title', 'text', 'text_sha256'])
            docs = docs[docs.doc_id.isin(REP_IDS)].sort_values('doc_id')
            assert (docs.text_sha256.values == docs.doc_id.map(REP_SHA).values).all()
            if SAMPLE_DOCS:
                docs = docs.sample(n=min(SAMPLE_DOCS, len(docs)), random_state=0).sort_values('doc_id')
            jobs = [(sha, int(doc_id), lang, clean_title(title), text) for doc_id, sha, title, text
                    in zip(docs.doc_id, docs.text_sha256, docs.title, docs.text)]
            del docs
            stats['read_seconds'] += time.time() - started
            started = time.time()
            rows = []
            for result in pool.imap(chunk_one, jobs, chunksize=64):
                ids = int(IDS_OF_SHA[result['text_sha256']])
                stats['texts'] += 1
                stats['doc_ids'] += ids
                stats['changed'] += result['changed']
                stats['changed_ids'] += ids if result['changed'] else 0
                stats['not_in_raw'] += result['not_in_raw']
                stats['chunk_counts'].append(len(result['chunks']))
                for order, start, end, chunk_text, units in result['chunks']:
                    stats['units'].append(units)
                    rows.append((f"{result['text_sha256'][:16]}-{order}", result['text_sha256'],
                                 result['rep_doc_id'], lang, order, start, end, chunk_text, units,
                                 result['title'], CLEAN_VERSION, CHUNK_PARAMS))
            stats['chunk_seconds'] += time.time() - started
            started = time.time()
            stats['chunks'] += write_part(lang, number, rows)
            stats['write_seconds'] += time.time() - started
            print(f'{lang} part-{number:03d}: {len(jobs):,} texts, {len(rows):,} chunks, '
                  f'chunk {stats["chunk_seconds"]:.1f}s total so far', flush=True)
            del jobs, rows
if not SAMPLE_DOCS:
    assert sum(stats['texts'] for stats in PER_LANG.values()) == len(groups)

## 4. Manifest

In [ ]:
def summarise(stats):
    counts, units = np.array(stats['chunk_counts']), np.array(stats['units'])
    return dict(
        texts=stats['texts'], doc_ids=stats['doc_ids'], chunks=stats['chunks'],
        texts_changed_by_cleaning=stats['changed'], doc_ids_changed_by_cleaning=stats['changed_ids'],
        chunks_not_in_raw=stats['not_in_raw'],
        chunks_per_text_mean=round(float(counts.mean()), 2),
        chunks_per_text_p50=int(np.percentile(counts, 50)), chunks_per_text_p90=int(np.percentile(counts, 90)),
        chunks_per_text_max=int(counts.max()),
        n_units_p50=int(np.percentile(units, 50)), n_units_p90=int(np.percentile(units, 90)),
        n_units_max=int(units.max()),
        read_seconds=round(stats['read_seconds'], 1), chunk_seconds=round(stats['chunk_seconds'], 1),
        write_seconds=round(stats['write_seconds'], 1),
        chunk_seconds_per_1000_texts=round(stats['chunk_seconds'] / stats['texts'] * 1000, 2),
        seconds_per_1000_texts=round((stats['read_seconds'] + stats['chunk_seconds'] + stats['write_seconds'])
                                     / stats['texts'] * 1000, 2))


SUMMARY = {lang: summarise(stats) for lang, stats in PER_LANG.items()}
MANIFEST_OUT = dict(
    dataset_version=MANIFEST['version'], dataset_built_at=MANIFEST['built_at'],
    clean_version=CLEAN_VERSION, chunk_params=CHUNK_PARAMS, id_scheme='{text_sha256[:16]}-{order}',
    excluded_ids=len(EXCLUDED_DOC_IDS), hash_groups=len(groups),
    group_types={name: int(count) for name, count in groups.group_type.value_counts().items()},
    sample_docs_per_part=SAMPLE_DOCS, languages=SUMMARY,
    elapsed_seconds=round(time.time() - T0, 1))
(OUT / 'chunk_manifest.json').write_text(json.dumps(MANIFEST_OUT, ensure_ascii=False, indent=2) + '\n')

## 5. Summary

In [ ]:
pd.DataFrame(SUMMARY).T